# Экперимент с Baseline++: base (ffn=1536) vs Group Query Attention vs Query-Key norm

Надо скопировать этот ноутбук и выполнить общие ячейки, затем выбрать **одну из трёх веток**: с нуля, с весов или resume. У каждой ветки свои параметры, сетап и запуск. Для последовательного запуска нескольких новых экспериментов предусмотрен отдельный блок очереди в конце; он выполняется вместо одиночной ветки.

Обучение запускается на двух GPU T4 в Kaggle. Метрики отправляются в W&B, результаты сохраняются в `/kaggle/working/experiment_runs`.

## Общие пути

- `PROJECT_PATH`: репозиторий с `utilities/train_script.py` и конфигами.
- `DATA_PATH`: исходная папка `tokenized` с `dataset_dict.json`, `tokenizer.json`, `train` и `val`. Используется при создании нового эксперимента и подготовке отсутствующих данных для resume.
- `WORK_DIR`: рабочая папка Kaggle для конфигов, данных и результатов.

Имя эксперимента и имя конфига задаются отдельно в ветках 1 и 2. Начальные веса нужны только ветке 2, папка сохранённого эксперимента — только ветке 3.

In [1]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized").expanduser().resolve()
WORK_DIR = Path("/kaggle/working").resolve()

TOKENIZER_PATH = DATA_PATH / "tokenizer.json"
DATA_DIR = WORK_DIR / "tokenized"

In [2]:
if not (PROJECT_PATH / "utilities" / "train_script.py").is_file():
    raise FileNotFoundError(f"Скрипт обучения не найден в репозитории: {PROJECT_PATH}")

print("Репозиторий:", PROJECT_PATH)
print("Исходные данные:", DATA_PATH)
print("Рабочая папка:", WORK_DIR)

Репозиторий: /kaggle/input/datasets/wwip62/nlp-hw2-codegen
Исходные данные: /kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized
Рабочая папка: /kaggle/working


In [3]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

Note: you may need to restart the kernel to use updated packages.


## Импорты

In [4]:
import os
import shutil
import subprocess
import sys

import torch
import yaml
from kaggle_secrets import UserSecretsClient

## Общие настройки

In [5]:
NUM_GPUS = 2

WANDB_PROJECT = "nlp-hw2"
WANDB_ENTITY = "62wip-hse-ml"

## Настройка W&B

In [6]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_MODE"] = "online"
env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Подготовка данных и нового эксперимента

`prepare_data` копирует валидацию в рабочую папку для записи кеша и подключает train ссылкой. Функция вызывается при новом эксперименте и при отсутствии данных для resume.

`prepare_train_config` используется только в ветках 1 и 2: создаёт конфиг с абсолютными путями и именем эксперимента. Параметры warmup переопределяются независимо; `None` сохраняет соответствующее значение исходного конфига.

In [7]:
def prepare_data(data_dir=DATA_DIR):
    dataset_file = DATA_PATH / "dataset_dict.json"
    if not dataset_file.is_file():
        raise FileNotFoundError(f"Файл данных не найден: {dataset_file}")
    for split in ("train", "val"):
        if not (DATA_PATH / split).is_dir():
            raise FileNotFoundError(f"Выборка не найдена: {DATA_PATH / split}")

    data_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA_PATH / "dataset_dict.json", data_dir / "dataset_dict.json")
    shutil.copytree(DATA_PATH / "val", data_dir / "val", dirs_exist_ok=True)

    train_path = data_dir / "train"
    if train_path.is_symlink():
        train_path.unlink()
    shutil.copytree(DATA_PATH / "train", train_path, dirs_exist_ok=True)


def prepare_train_config(run_name, config_name):
    config_source = PROJECT_PATH / "experiment_configs" / config_name
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}. Нужно другое RUN_NAME или режим resume.")
    config = yaml.safe_load(config_source.read_text())
    if not TOKENIZER_PATH.is_file():
        raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
    prepare_data()

    config["name"] = run_name
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    if LR is not None:
        config["optimizer"]["params"]["lr"] = LR
    if MIN_LR is not None:
        config["scheduler"]["params"]["eta_min"] = MIN_LR
    if WARMUP_STEPS is not None or WARMUP_START_FACTOR is not None:
        warmup = config["scheduler"].get("warmup") or {}
        if WARMUP_STEPS is not None:
            warmup["steps"] = WARMUP_STEPS
        if WARMUP_START_FACTOR is not None:
            warmup["start_factor"] = WARMUP_START_FACTOR
        config["scheduler"]["warmup"] = warmup
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))

    print("Конфиг:", config_path)
    print("Данные:", DATA_DIR)
    print("Токенайзер:", TOKENIZER_PATH)
    print("Результаты:", run_path)
    return config_path, run_path

## Запуск 2х экспериментов с лимитом времени

Обучаем с нуля два варианта Baseline++ с бюджетом по часу:
- **GQA:** используем 8 query-голов и 2 KV-головы.
- **QK-norm:** добавляем нормализацию query и key, сохраняя обычный attention с 8 KV-головами.

В обоих вариантах — 5 encoder-слоёв и 7 decoder-слоёв. Сравниваем pass@1 на одной выборке из 512 вопросов. В час входят обучение, валидации и сохранения.

In [8]:
CONFIG_PATHS = [
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_gqa.yaml",
    PROJECT_PATH / "experiment_configs/1_hour_exps/pilot_1h_qk_norm.yaml",
]
MAX_TRAIN_SECONDS = 3600

In [9]:
if not CONFIG_PATHS:
    raise ValueError("CONFIG_PATHS должен содержать хотя бы один путь к конфигу.")

experiments = []
run_names = set()
for source_path in CONFIG_PATHS:
    source_path = Path(source_path).expanduser().resolve()
    config = yaml.safe_load(source_path.read_text())
    run_name = config["name"]
    if not isinstance(run_name, str) or not run_name:
        raise ValueError(f"В конфиге не задано имя эксперимента: {source_path}")
    if run_name in run_names:
        raise ValueError(f"Имя эксперимента повторяется в очереди: {run_name}")
    run_names.add(run_name)

    run_path = WORK_DIR / "experiment_runs" / run_name
    if run_path.exists():
        raise FileExistsError(f"Эксперимент уже существует: {run_path}")
    config["data"]["tokenized_dir"] = str(DATA_DIR)
    config["data"]["tokenizer_path"] = str(TOKENIZER_PATH)
    config_path = WORK_DIR / f"{run_name}_config.yaml"
    experiments.append((config_path, config, run_path))

if not TOKENIZER_PATH.is_file():
    raise FileNotFoundError(f"Токенайзер не найден: {TOKENIZER_PATH}")
prepare_data()

for config_path, config, run_path in experiments:
    config_path.write_text(yaml.safe_dump(config, sort_keys=False))
    print(f"{config['name']}: {config_path} → {run_path}")

print(f"В очереди: {len(experiments)}; бюджет на эксперимент: {MAX_TRAIN_SECONDS} с")

pilot_1h_gqa: /kaggle/working/pilot_1h_gqa_config.yaml → /kaggle/working/experiment_runs/pilot_1h_gqa
pilot_1h_qk_norm: /kaggle/working/pilot_1h_qk_norm_config.yaml → /kaggle/working/experiment_runs/pilot_1h_qk_norm
В очереди: 2; бюджет на эксперимент: 3600 с


In [10]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

for number, (config_path, config, run_path) in enumerate(experiments, start=1):
    print(f"[{number}/{len(experiments)}] Запуск: {config['name']}", flush=True)
    command = [
        sys.executable, "-u", "-m", "torch.distributed.run",
        "--standalone", f"--nproc_per_node={NUM_GPUS}",
        "-m", "utilities.train_script",
        "--device", "cuda",
        "--config", str(config_path),
        "--max-train-seconds", str(MAX_TRAIN_SECONDS),
        "--wandb-project", WANDB_PROJECT,
    ]
    if WANDB_ENTITY:
        command += ["--wandb-entity", WANDB_ENTITY]

    result = subprocess.run(
        command,
        cwd=PROJECT_PATH,
        env={**env, "PROJECT_DIR": str(WORK_DIR)},
        check=False,
    )
    if result.returncode != 0:
        print(f"Очередь остановлена: {config['name']}, код выхода {result.returncode}.", flush=True)
        break
    print(f"Завершён: {config['name']} -> {run_path}", flush=True)
else:
    print("Все эксперименты в очереди завершены.")

[1/2] Запуск: pilot_1h_gqa



*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Flattening the indices: 100%|██████████| 512/512 [00:00<00:00, 37851.79 examples/s]


Run: /kaggle/working/experiment_runs/pilot_1h_gqa; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/pilot_1h_gqa/wandb/run-20261023_230146-p8242soc
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pilot_1h_gqa
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/p8242soc


Epoch 1, mini-epoch 1 | train loss = 3.2741, val loss = 2.2734
Examples: 100,020 | time = 249.2 s (train 199.8, val 49.4) | train/example = 0.001998 s
Small validation: token accuracy = 62.39%, top-5 = 77.91%
Heavy validation: pass@1 = 1.76% (9 / 512) | time = 47.9 s
Syntax valid = 87.70%, numeric return = 71.48%, no EOS = 6.64%
BLEU = 27.45, repetition = 27.27%, timeout = 0.00%
Epoch 1, mini-epoch 2 | train loss = 1.5036, val loss = 1.0446
Examples: 200,110 | time = 245.0 s (train 198.5, val 46.4) | train/example = 0.001984 s
Small validation: token accuracy = 78.51%, top-5 = 91.69%
Heavy validation: pass@1 = 11.13% (57 / 512) | time = 45.1 s
Syntax valid = 99.80%, numeric return = 90.82%, no EOS = 3.91%
BLEU = 35.62, repetition = 15.00%, timeout = 0.00%
Epoch 1, mini-epoch 3 | train loss = 0.9175, val loss = 0.7539
Examples: 300,009 | time = 251.7 s (train 201.5, val 50.2) | train/example = 0.002017 s
Small validation: token accuracy = 82.88%, top-5 = 94.62%
Heavy validation: pass@1 

...

Run: /kaggle/working/experiment_runs/pilot_1h_qk_norm; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/pilot_1h_qk_norm/wandb/run-20261024_000327-xwf3gtnq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run pilot_1h_qk_norm
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/xwf3gtnq


Epoch 1, mini-epoch 1 | train loss = 3.1523, val loss = 2.1978
Examples: 100,020 | time = 271.1 s (train 220.8, val 50.3) | train/example = 0.002208 s
Small validation: token accuracy = 63.85%, top-5 = 79.26%
Heavy validation: pass@1 = 1.17% (6 / 512) | time = 49.0 s
Syntax valid = 93.55%, numeric return = 76.95%, no EOS = 5.47%
BLEU = 28.78, repetition = 23.67%, timeout = 0.00%
Epoch 1, mini-epoch 2 | train loss = 1.4752, val loss = 1.0321
Examples: 200,110 | time = 261.7 s (train 215.5, val 46.2) | train/example = 0.002153 s
Small validation: token accuracy = 78.63%, top-5 = 91.82%
Heavy validation: pass@1 = 12.50% (64 / 512) | time = 44.9 s
Syntax valid = 99.80%, numeric return = 92.38%, no EOS = 3.52%
BLEU = 36.60, repetition = 15.35%, timeout = 0.00%
Epoch 1, mini-epoch 3 | train loss = 0.9032, val loss = 0.7392
Examples: 300,009 | time = 270.6 s (train 221.1, val 49.5) | train/example = 0.002213 s
Small validation: token accuracy = 83.05%, top-5 = 94.73%
Heavy validation: pass@1 

wandb: updating run metadata
wandb: uploading config.yaml
wandb: 
wandb: Run history:
wandb:              batch_index ▁▂▂▃▃▄▄▅▅▆▇▇██
wandb:             batches_seen ▁▂▂▃▃▄▄▅▅▆▇▇██
wandb:                     bleu ▁▃▅▆▇▇▇▇▇▇▇██▇
wandb:                  correct ▁▂▄▅▅▆▆▇▇▇▇▇██
wandb:                    epoch ▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:            examples_seen ▁▂▂▃▃▄▄▅▅▆▇▇██
wandb: heavy_validation_seconds ▇▃▇█▆▆█▄▇▁▅▆▅▁
wandb:                       lr ▁█████████████
wandb:               mini_epoch ▁▂▂▃▃▄▄▅▅▆▆▇▇█
wandb:      mini_epoch_examples █████████████▁
wandb:                      +26 ...
wandb: 
wandb: Run summary:
wandb:              batch_index 10603
wandb:             batches_seen 10603
wandb:                     bleu 49.12953
wandb:           bleu_signature nrefs:var|case:mixed...
wandb:                  correct 266
wandb:                    epoch 1
wandb:            examples_seen 1352611
wandb: heavy_validation_seconds 43.73353
wandb:                       lr 0.0003
wandb:           

Завершён: pilot_1h_qk_norm -> /kaggle/working/experiment_runs/pilot_1h_qk_norm
Все эксперименты в очереди завершены.


## Удаление рабочей копии данных

In [11]:
cleanup_dir = WORK_DIR / "tokenized"
if cleanup_dir.is_symlink():
    cleanup_dir.unlink()
elif cleanup_dir.exists():
    shutil.rmtree(cleanup_dir)
print("Рабочая копия данных удалена:", cleanup_dir)

Рабочая копия данных удалена: /kaggle/working/tokenized
